# Model training

### Import data and required packages

In [121]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

In [122]:
df = pd.read_csv('../../data/clean/clean_public_data.csv')

In [123]:
df.head()

,epglnren,sup_riscaldata,sup_raffrescata,vol_riscaldato,vol_raffrescato,sup_disperdente,rapsv,asolsut,presenza_clim_invernale,presenza_clim_estiva,...,tipologia_costruttiva,num_servizi,efficienza_media,potenza_tot,num_simulati,construction_era,piano,tip_edilizia_sq,zona_clim_num,zona_clim_gg_mid
0,196.28,210.07,0.00,865.20,0.00,540.37,0.6246,0.0503,True,False,...,muratura portante,2,0.835000,54.80,0,1,multi_floor,1,4,1750
1,193.17,72.29,72.29,261.13,261.13,176.77,0.6769,0.0550,True,True,...,c.a. con laterizi,3,0.600000,29.62,0,3,ground,25,4,1750
2,173.49,47.90,0.00,202.85,0.00,108.34,0.5341,0.0412,True,False,...,muratura portante,2,0.515000,1.50,1,1,ground,9,5,2550
3,177.51,60.00,60.00,217.08,217.08,171.90,0.7918,0.0340,True,True,...,legno,3,0.716667,13.00,0,1,underground,64,4,1750
4,173.88,194.01,0.00,785.11,0.00,523.24,0.6665,0.0692,True,False,...,muratura portante,2,0.790000,48.20,0,1,multi_floor,1,5,2550


### Define X and y then preprocess

In [124]:
X = df.drop(['epglnren'], axis=1) 

In [125]:
y = df['epglnren']


In [126]:
# Column transformer
numerical_features = X.select_dtypes(include=['int64', 'float64', 'bool']).columns
categorical_features = X.select_dtypes(include=['object', 'category', 'str']).columns

print(f'features excluding target variable: {len(X.columns)}')
print(f'numerical features: {len(numerical_features)}')
print(f'categorical features: {len(categorical_features)}')

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder()

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", categorical_transformer, categorical_features),
        ("StandardScaler", numeric_transformer, numerical_features)
    ]
)

features excluding target variable: 25
numerical features: 21
categorical features: 4


In [127]:
X = preprocessor.fit_transform(X)

In [128]:
X.shape

(8882, 52)

In [129]:
# train test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

((7105, 52), (1777, 52))

### Create an evaluation function to present all metrics after model training

In [130]:
def evaluate_model(true, predicted):

    mse = mean_squared_error(true, predicted)
    mae = mean_absolute_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2 = r2_score(true, predicted)
    
    return mae, mse, rmse, r2

In [131]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Lasso Regression": Lasso(),
    "KNN Regressor": KNeighborsRegressor(),
    "Decision Tree Regressor": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "AdaBoost Regressor": AdaBoostRegressor(),
    "SVR": SVR(),
    "CatBoost Regressor": CatBoostRegressor(verbose=0),
    "XGBoost Regressor": XGBRegressor(verbose=0),
    "LightGBM Regressor": LGBMRegressor(verbose=0)
}
model_results = []
r2_scores = []

for i in range(len(list(models))):

    model = list(models.values())[i]
    model.fit(X_train, y_train) # train the model

    # make predictions
    y_train_pred = model.predict(X_train) 
    y_test_pred = model.predict(X_test)

    # evaluate model performance
    model_train_mae, model_train_mse, model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)
    model_test_mae, model_test_mse, model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    print(list(models.keys())[i])
    model_results.append(list(models.keys())[i])

    print('\nModel performance on training set:')
    print(f'  MAE: {model_train_mae:.4f}')
    print(f'  MSE: {model_train_mse:.4f}')
    print(f'  RMSE: {model_train_rmse:.4f}')
    print(f'  R2: {model_train_r2:.4f}')
    print ('---' * 10)
    print('Model performance on test set:')
    print(f'  MAE: {model_test_mae:.4f}')
    print(f'  MSE: {model_test_mse:.4f}')
    print(f'  RMSE: {model_test_rmse:.4f}')
    print(f'  R2: {model_test_r2:.4f}')
    
    r2_scores.append(model_test_r2)

    print('=' * 20)
    print('\n')

Linear Regression

Model performance on training set:
  MAE: 57.8108
  MSE: 6813.5557
  RMSE: 82.5443
  R2: 0.4222
------------------------------
Model performance on test set:
  MAE: 56.1007
  MSE: 6331.5012
  RMSE: 79.5707
  R2: 0.4577


Ridge Regression

Model performance on training set:
  MAE: 57.8104
  MSE: 6813.8528
  RMSE: 82.5461
  R2: 0.4222
------------------------------
Model performance on test set:
  MAE: 56.0771
  MSE: 6325.4214
  RMSE: 79.5325
  R2: 0.4582


Lasso Regression

Model performance on training set:
  MAE: 58.4950
  MSE: 6982.4068
  RMSE: 83.5608
  R2: 0.4079
------------------------------
Model performance on test set:
  MAE: 56.4114
  MSE: 6453.1862
  RMSE: 80.3317
  R2: 0.4473


KNN Regressor

Model performance on training set:
  MAE: 46.1338
  MSE: 4598.4964
  RMSE: 67.8122
  R2: 0.6101
------------------------------
Model performance on test set:
  MAE: 55.9833
  MSE: 6597.7674
  RMSE: 81.2266
  R2: 0.4349


Decision Tree Regressor

Model performance on 

c:\Users\AmirN\AppData\Local\Programs\Python\Python314\Lib\site-packages\xgboost\training.py:200: UserWarning: [15:26:33] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "verbose" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\AmirN\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\AmirN\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


LightGBM Regressor

Model performance on training set:
  MAE: 38.0593
  MSE: 2867.1814
  RMSE: 53.5461
  R2: 0.7569
------------------------------
Model performance on test set:
  MAE: 49.5031
  MSE: 5227.7205
  RMSE: 72.3030
  R2: 0.5523




In [132]:
pd.DataFrame(list(zip(model_results, r2_scores)), columns=['Model', 'R2 Score']).sort_values(by='R2 Score', ascending=False)

,Model,R2 Score
8,CatBoost Regressor,0.584953
10,LightGBM Regressor,0.552261
9,XGBoost Regressor,0.539820
5,Random Forest Regressor,0.515908
1,Ridge Regression,0.458246
0,Linear Regression,0.457725
2,Lasso Regression,0.447303
3,KNN Regressor,0.434920
7,SVR,0.315412
6,AdaBoost Regressor,0.038095
